In [1]:
# Component I. Vector Indexing and Embedding

In [ ]:
## Step I-1 — Install the Required Libraries

**Purpose:** Install every Python package the pipeline needs inside the Colab runtime.

**How it works:**
1. The comment banner at the top names the whole component: PDF extraction → chunking → embeddings → FAISS → retrieval, with an Excel file for inspecting the chunks.
2. The `!` prefix tells Colab to run the line as a **shell command** instead of Python code.
3. `pip install -q` installs the packages quietly (`-q` hides most of the log). The trailing `\` characters continue one command across several lines.
4. The packages and their roles:
   - `faiss-cpu`: Facebook AI Similarity Search, the library that stores vectors and finds the closest ones (CPU build).
   - `sentence-transformers`: loads the embedding model that turns text into vectors.
   - `pypdf`: reads PDF files and extracts their text layer.
   - `transformers`: Hugging Face library, used in Component II to load the FLAN-T5 language model.
   - `sentencepiece`: the tokenizer backend that T5 models need.
   - `pandas`: builds the table of chunks.
   - `openpyxl`: writes and formats the `.xlsx` workbook.
5. Packages that Colab already has are reported as "already satisfied" and skipped.

**Output:** Little or no text. Run it once per Colab session, because a runtime reset removes installed packages.

In [ ]:
# ============================================================
# RAG DOCUMENT PROCESSING PIPELINE
# PDF EXTRACTION, CHUNKING, EMBEDDINGS, FAISS, AND RETRIEVAL
# WITH DOWNLOADABLE EXCEL CHUNK INSPECTION
# Google Colab implementation
# ============================================================


# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    transformers \
    sentencepiece \
    pandas \
    openpyxl

## Step I-2 — Import the Libraries

**Purpose:** Load the installed packages into memory so the later cells can use them.

**How it works:**
1. `from google.colab import files` gives `files.upload()` (browser file picker) and `files.download()` (send a file to your computer). This module exists only in Google Colab.
2. `PdfReader` (pypdf) opens a PDF and exposes its pages.
3. `SentenceTransformer` loads the embedding model in Step I-12.
4. Standard helpers:
   - `faiss`: the vector index.
   - `numpy` (`np`): arrays and the average chunk length.
   - `pandas` (`pd`): the Excel table.
   - `pickle`: saves Python objects to disk.
   - `re`: regular expressions, used for whitespace cleanup.
   - `os`: file-name handling.
5. `Font`, `PatternFill`, and `Alignment` from `openpyxl.styles` style the Excel header row in Step I-10.

**Output:** Nothing is printed. A `ModuleNotFoundError` here means Step I-1 was not run in this session.

In [ ]:
# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np
import pandas as pd
import pickle
import re
import os

from openpyxl.styles import Font, PatternFill, Alignment

## Step I-3 — Set the Indexing Parameters

**Purpose:** Define the experiment settings in one place.

**How it works:**
1. `CHUNK_SIZE = 1000`: each chunk holds at most 1,000 **characters** (letters, digits, spaces and punctuation), not words or tokens.
2. `CHUNK_OVERLAP = 200`: consecutive chunks from the same page share 200 characters. The window therefore moves forward by `1000 − 200 = 800` characters each time (see Step I-8). The overlap keeps a sentence cut at a chunk border fully readable in at least one chunk.
3. `TOP_K = 3`: how many of the most similar chunks are retrieved for each question (used in Steps I-18 and II-9).
4. `EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"`: the Sentence-Transformers model that converts text into **embeddings**. An embedding is a list of numbers (a vector) that represents the meaning of the text; this model produces 384 numbers per text.

**Experiment tip (PT-F1):** To test another configuration, change `CHUNK_SIZE` and/or `CHUNK_OVERLAP` here (1000/200 is the baseline). Then re-run this cell and every cell after it.

In [ ]:
# ============================================================
# 3. INDEXING PARAMETERS
# ============================================================

# Change these values to experiment with chunking.

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 3

# Embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

## Step I-4 — Validate the Chunk Parameters and Show the Configuration

**Purpose:** Stop early with a clear error if the settings in Step I-3 cannot work.

**How it works:**
1. `CHUNK_SIZE <= 0` → `ValueError`, because a chunk must contain at least one character.
2. `CHUNK_OVERLAP < 0` → `ValueError`, because negative overlap has no meaning.
3. `CHUNK_OVERLAP >= CHUNK_SIZE` → `ValueError`. The step size `CHUNK_SIZE − CHUNK_OVERLAP` would be zero or negative, so the chunking loop in Step I-8 would never move forward and would run forever.
4. `TOP_K <= 0` → `ValueError`, because retrieval must return at least one chunk.
5. If every check passes, the cell prints the configuration banner with the chunk size, overlap, Top-K and embedding model.

**Output:** The `RAG INDEXING CONFIGURATION` summary.

In [ ]:
# ============================================================
# 4. VALIDATE CHUNK PARAMETERS
# ============================================================

if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be greater than 0.")

if CHUNK_OVERLAP < 0:
    raise ValueError("CHUNK_OVERLAP cannot be negative.")

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError(
        "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
    )

if TOP_K <= 0:
    raise ValueError("TOP_K must be greater than 0.")


print("=" * 70)
print("RAG INDEXING CONFIGURATION")
print("=" * 70)

print("Chunk size       :", CHUNK_SIZE)
print("Chunk overlap    :", CHUNK_OVERLAP)
print("Top-K             :", TOP_K)
print("Embedding model   :", EMBEDDING_MODEL_NAME)

## Step I-5 — Upload the PDF Document

**Purpose:** Get the source PDF into the Colab runtime.

**How it works:**
1. `files.upload()` opens a file-picker button under the cell. The selected file is saved to the Colab working folder (`/content`). The function returns a dictionary of the form `{file_name: file_bytes}`.
2. If the dictionary is empty (nothing chosen), the cell raises a `RuntimeError`.
3. `next(iter(uploaded.keys()))` takes the **first** file name. If several files are uploaded, only the first is used.
4. `.lower().endswith(".pdf")` checks the extension without regard to letter case. A non-PDF file raises a `ValueError`.
5. The file name is kept in `pdf_file`. Later steps use it for reading the text, naming the Excel file and saving the configuration.

**Output:** The name of the selected file. The cell waits until you pick a file.

In [ ]:
# ============================================================
# 5. UPLOAD PDF DOCUMENT
# ============================================================

print("\n")
print("=" * 70)
print("UPLOAD DOCUMENT")
print("=" * 70)

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No file was uploaded.")

# Use the first uploaded PDF
pdf_file = next(iter(uploaded.keys()))

if not pdf_file.lower().endswith(".pdf"):
    raise ValueError(
        "The uploaded file is not a PDF. Please upload a PDF."
    )

print("\nSelected file:")
print(pdf_file)

## Step I-6 — Extract the Text from the PDF

**Purpose:** Read the text of every page and clean its whitespace.

**How it works:**
1. `PdfReader(pdf_file)` parses the uploaded PDF.
2. `enumerate(reader.pages, start=1)` loops over the pages and numbers them from 1, the way a reader counts pages.
3. `page.extract_text()` returns the page's text layer. It returns an empty result for pages that are only images, such as scans.
4. `re.sub(r"\s+", " ", page_text)` replaces every run of whitespace (spaces, tabs, line breaks) with a single space. `.strip()` removes spaces at both ends. Each page therefore becomes one continuous line of text.
5. Pages that still contain text are stored in the `pages` list as `{"page": <number>, "text": <cleaned text>}`. Empty pages are skipped.

**Output:** The total number of PDF pages and how many of them contain text.

In [ ]:
# ============================================================
# 6. EXTRACT TEXT FROM PDF
# ============================================================

print("\n")
print("=" * 70)
print("EXTRACTING PDF TEXT")
print("=" * 70)

reader = PdfReader(pdf_file)

pages = []

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if page_text:

        # Normalize whitespace
        page_text = re.sub(
            r"\s+",
            " ",
            page_text
        ).strip()

        if page_text:
            pages.append({
                "page": page_number,
                "text": page_text
            })


print("PDF pages:", len(reader.pages))
print("Pages containing text:", len(pages))

## Step I-7 — Check Whether Text Was Extracted

**Purpose:** Stop the pipeline if the PDF produced no usable text.

**How it works:**
1. If the `pages` list from Step I-6 is empty, the cell raises a `RuntimeError`.
2. The message explains the usual cause: the PDF is made of scanned images. Such a PDF needs **OCR** (optical character recognition) before its text can be read.
3. Without this check, the later steps would fail with less obvious errors, for example when building an index with no vectors.

**Output:** Nothing when text was found.

In [ ]:
# ============================================================
# 7. CHECK WHETHER TEXT WAS EXTRACTED
# ============================================================

if len(pages) == 0:
    raise RuntimeError(
        "No text could be extracted from the PDF. "
        "The PDF may consist of scanned images and require OCR."
    )

## Step I-8 — Create the Document Chunks

**Purpose:** Split each page into overlapping, fixed-size pieces of text (**chunks**). Each chunk is the unit that gets embedded and retrieved.

**How it works (`create_chunks`):**
1. `step_size = chunk_size − chunk_overlap` (800 with the default settings).
2. The function processes every page **separately**, so a chunk never crosses a page boundary and overlap exists only between chunks of the same page.
3. For each page, a sliding window starts at `start = 0`:
   - `end = start + chunk_size`
   - `page_text[start:end].strip()` cuts out the chunk. Python slicing stops at the end of the text automatically.
   - Non-empty chunks are appended as `{"chunk_id", "page", "text"}`. `chunk_id` is a running counter across the whole document (0, 1, 2, …).
   - `start += step_size` moves the window forward while keeping the overlap.
   - The loop repeats while `start < len(page_text)`.
4. The cut is purely character-based, so a chunk may begin or end in the middle of a word or sentence.
5. After the definition, the function is called with the Step I-3 settings and the result is stored in `chunks`.

**Note:** Because the loop runs while `start` is still inside the page, the last window of a page can be a short tail that is already fully contained in the previous chunk. For example, a 1,622-character page gives `[0:1000]`, `[800:1622]` and a 22-character tail `[1600:1622]`. This explains very short values in the "Shortest chunk" statistic of Step I-9. The original behavior is kept unchanged.

**Output:** Nothing is printed. The `chunks` list is used by every later step.

In [ ]:
# ============================================================
# 8. CREATE DOCUMENT CHUNKS
# ============================================================

def create_chunks(
    pages,
    chunk_size,
    chunk_overlap
):
    """
    Split each page into character-based chunks.

    Chunking is performed independently for each page.
    Overlap is maintained between consecutive chunks
    from the same page.
    """

    chunks = []
    chunk_id = 0

    step_size = chunk_size - chunk_overlap

    for page_data in pages:

        page_number = page_data["page"]
        page_text = page_data["text"]

        start = 0

        while start < len(page_text):

            end = start + chunk_size

            chunk_text = page_text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "chunk_id": chunk_id,
                    "page": page_number,
                    "text": chunk_text
                })

                chunk_id += 1

            # Move forward while maintaining overlap
            start += step_size

    return chunks


chunks = create_chunks(
    pages,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

## Step I-9 — Display the Chunking Information

**Purpose:** Summarize the chunks so different chunk settings can be compared.

**How it works:**
1. Prints the number of chunks created.
2. If no chunks exist, the cell raises a `RuntimeError` as a safety net.
3. A list comprehension builds `chunk_lengths`, the character count of every chunk.
4. Prints three statistics:
   - average length: `np.mean`, rounded to 2 decimals
   - shortest chunk: `min`
   - longest chunk: `max`

**How to read it:** The longest chunk can never exceed `CHUNK_SIZE`. An average well below `CHUNK_SIZE` means many pages are short or there are many tail chunks (see the note in Step I-8).

**Output:** The `CHUNKING RESULTS` block.

In [ ]:
# ============================================================
# 9. DISPLAY CHUNKING INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("CHUNKING RESULTS")
print("=" * 70)

print("Number of chunks:", len(chunks))

if len(chunks) == 0:
    raise RuntimeError(
        "No chunks were generated. Check the extracted PDF text."
    )

chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print(
    "Average characters per chunk:",
    round(np.mean(chunk_lengths), 2)
)

print(
    "Shortest chunk:",
    min(chunk_lengths),
    "characters"
)

print(
    "Longest chunk:",
    max(chunk_lengths),
    "characters"
)

## Step I-10 — Export the Generated Chunks to Excel

**Purpose:** Save every chunk into a formatted Excel workbook for manual inspection, then download it.

**How it works:**
1. **Build the rows.** Each chunk becomes a dictionary with these columns: `Chunk_ID`, `Source_Document`, `Page_Number`, `Character_Count`, `Chunk_Size_Setting`, `Chunk_Overlap_Setting` and `Chunk_Text`. The list is turned into a pandas `DataFrame` (`chunks_df`).
2. **Name the file.** `os.path.basename` removes any folder path and `os.path.splitext(...)[0]` removes the `.pdf` extension, giving `<pdf name>_chunks_inspection.xlsx`.
3. **Write the sheet.** `pd.ExcelWriter(..., engine="openpyxl")` opens the workbook. `to_excel` writes the table to the sheet `Generated_Chunks` without the pandas index column.
4. **Format the sheet** through the openpyxl worksheet object `writer.sheets["Generated_Chunks"]`:
   - Header row 1: bold white font, dark-blue fill (`1F4E78`), centered, wrapped text, row height 32.
   - Column widths A–G are set individually. Column G (`Chunk_Text`) is the widest at 80.
   - Data rows are top-aligned, and only column 7 (G, the chunk text) wraps.
   - `freeze_panes = "A2"` keeps the header visible while scrolling.
   - `auto_filter.ref = worksheet.dimensions` adds filter drop-downs over the whole used range.
5. The file is saved when the `with` block ends.
6. Prints the file name, the number of rows exported, the sheet name and a preview of the first 10 rows (ID, page and character count).
7. `files.download(EXCEL_FILE)` sends the workbook to your computer through the browser.

**Output:** The export summary and preview, plus a file download.

In [ ]:
# ============================================================
# 10. EXPORT GENERATED CHUNKS TO EXCEL
# ============================================================

print("\n")
print("=" * 70)
print("EXPORTING GENERATED CHUNKS TO EXCEL")
print("=" * 70)

# Prepare one Excel row for every generated chunk.
chunk_records = []

for chunk in chunks:

    chunk_records.append({
        "Chunk_ID": chunk["chunk_id"],
        "Source_Document": pdf_file,
        "Page_Number": chunk["page"],
        "Character_Count": len(chunk["text"]),
        "Chunk_Size_Setting": CHUNK_SIZE,
        "Chunk_Overlap_Setting": CHUNK_OVERLAP,
        "Chunk_Text": chunk["text"]
    })

chunks_df = pd.DataFrame(chunk_records)

# Create an output filename based on the source PDF.
base_name = os.path.splitext(
    os.path.basename(pdf_file)
)[0]

EXCEL_FILE = f"{base_name}_chunks_inspection.xlsx"

# Create and format the Excel workbook.
with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    chunks_df.to_excel(
        writer,
        sheet_name="Generated_Chunks",
        index=False
    )

    worksheet = writer.sheets["Generated_Chunks"]

    # Format the header row.
    for cell in worksheet[1]:

        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # Set column widths.
    column_widths = {
        "A": 12,
        "B": 35,
        "C": 14,
        "D": 18,
        "E": 20,
        "F": 23,
        "G": 80
    }

    for column, width in column_widths.items():
        worksheet.column_dimensions[column].width = width

    # Format rows for easier inspection.
    for row in worksheet.iter_rows(
        min_row=2,
        max_row=worksheet.max_row
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=(cell.column == 7)
            )

    # Keep headers visible while scrolling.
    worksheet.freeze_panes = "A2"

    # Enable filtering.
    worksheet.auto_filter.ref = worksheet.dimensions

    # Set header height.
    worksheet.row_dimensions[1].height = 32

print("Excel file created:", EXCEL_FILE)
print("Chunks exported:", len(chunks_df))
print("Excel worksheet: Generated_Chunks")

print("\nExcel export preview:")

print(
    chunks_df[
        [
            "Chunk_ID",
            "Page_Number",
            "Character_Count"
        ]
    ].head(10).to_string(index=False)
)

# Automatically download the Excel workbook in Google Colab.
files.download(EXCEL_FILE)

## Step I-11 — Display Sample Chunks

**Purpose:** Show a few chunks in full so you can check the chunk quality by eye.

**How it works:**
1. `chunks[:3]` takes the first three chunks.
2. For each one, the cell prints its ID, page number, character count, a divider line and the full chunk text.

**What to look for:** When chunk 0 and chunk 1 come from the same page, the last ~200 characters of chunk 0 reappear at the start of chunk 1. That repeated text is the overlap. Also check whether the cuts split words or sentences.

**Output:** The `SAMPLE CHUNKS` block.

In [ ]:
# ============================================================
# 11. DISPLAY SAMPLE CHUNKS
# ============================================================

print("\n")
print("=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)

for chunk in chunks[:3]:

    print(f"\nChunk ID : {chunk['chunk_id']}")
    print(f"Page     : {chunk['page']}")
    print(f"Characters: {len(chunk['text'])}")

    print("-" * 70)

    print(chunk["text"])

## Step I-12 — Load the Embedding Model

**Purpose:** Load the model that converts text into vectors.

**How it works:**
1. `SentenceTransformer(EMBEDDING_MODEL_NAME)` downloads `all-MiniLM-L6-v2` from the Hugging Face Hub the first time it runs, then loads it from the local cache.
2. The model is a small 6-layer transformer (MiniLM). It maps any sentence or paragraph to a **384-dimensional** vector. Texts with similar meaning get vectors that point in similar directions.
3. The model reads at most **256 word-piece tokens** per text and ignores anything beyond that. A token is a word or part of a word. For English, a 1,000-character chunk is usually within this limit. Much larger chunks, such as 1,500 characters or more, may be partly ignored when embedded.
4. The model is stored in `embedding_model`. It is used for the chunks (Step I-14) and again for every question (Step I-17).

**Output:** The model name.

In [ ]:
# ============================================================
# 12. LOAD EMBEDDING MODEL
# ============================================================

print("\n")
print("=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)

## Step I-13 — Extract the Chunk Text

**Purpose:** Prepare a plain list of strings for the embedding model.

**How it works:**
1. A list comprehension takes the `"text"` value from each chunk dictionary and stores it in `chunk_texts`.
2. The order is the same as in `chunks`. This matters: position *i* in `chunk_texts` becomes row *i* of the embedding matrix, which becomes vector ID *i* in FAISS, which maps back through `chunk_lookup[i]` in Step I-16.

**Output:** Nothing is printed.

In [ ]:
# ============================================================
# 13. EXTRACT CHUNK TEXT
# ============================================================

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

## Step I-14 — Create the Document Embeddings

**Purpose:** Convert every chunk into a vector.

**How it works:**
1. `embedding_model.encode(chunk_texts, ...)` processes the chunks in batches. The arguments:
   - `convert_to_numpy=True` returns a NumPy array instead of a Python list.
   - `normalize_embeddings=True` rescales every vector to length 1 (unit L2 norm), so the inner product used in Step I-15 equals cosine similarity.
   - `show_progress_bar=True` shows a progress bar.
2. `.astype("float32")` converts the numbers to 32-bit floats, the data type FAISS requires.
3. Prints the matrix shape `(number_of_chunks, 384)`: one row per chunk and one column per embedding dimension.

**Output:** A progress bar and the embedding matrix shape.

In [ ]:
# ============================================================
# 14. CREATE DOCUMENT EMBEDDINGS
# ============================================================

print("\n")
print("=" * 70)
print("CREATING DOCUMENT EMBEDDINGS")
print("=" * 70)

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# FAISS expects float32 vectors.
embeddings = embeddings.astype("float32")

print("\nEmbedding matrix shape:")
print(embeddings.shape)

## Step I-15 — Create the FAISS Vector Store

**Purpose:** Store all chunk vectors in a searchable index (the **vector store**).

**How it works:**
1. `embedding_dimension = embeddings.shape[1]` reads the vector length (384).
2. `faiss.IndexFlatIP(embedding_dimension)` creates a **flat** index scored by **Inner Product**:
   - "Flat" means exact, brute-force search. Every stored vector is compared with the query, and no training or approximation is involved.
   - Because the vectors were normalized in Step I-14, the inner product equals **cosine similarity**. Scores run from −1 to 1, and higher means more similar.
3. `index.add(embeddings)` stores the vectors. FAISS gives them IDs 0, 1, 2, … in insertion order.
4. Prints the dimension and `index.ntotal`, the number of stored vectors. It should equal the number of chunks.

**Output:** The `CREATING FAISS VECTOR STORE` block.

In [ ]:
# ============================================================
# 15. CREATE FAISS VECTOR STORE
# ============================================================

print("\n")
print("=" * 70)
print("CREATING FAISS VECTOR STORE")
print("=" * 70)

embedding_dimension = embeddings.shape[1]

# Inner Product on normalized vectors is equivalent
# to cosine similarity.
index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors stored:", index.ntotal)

## Step I-16 — Create the Chunk Lookup

**Purpose:** Map FAISS search results back to the original chunk data.

**How it works:**
1. FAISS returns only integer positions, not text. The dictionary comprehension builds `chunk_lookup = {position: chunk}` for every chunk.
2. Given a FAISS result position, `chunk_lookup[position]` returns the chunk's ID, page and text.
3. Here `chunk_id` happens to equal the position, because both start at 0 and increase by one. The lookup keeps the mapping explicit and correct anyway.

**Output:** Nothing is printed.

In [ ]:
# ============================================================
# 16. CREATE CHUNK LOOKUP
# ============================================================

# The embedding row index corresponds to the position
# of the chunk in the chunks list.

chunk_lookup = {
    position: chunk
    for position, chunk in enumerate(chunks)
}

## Step I-17 — Define the Retrieval Function

**Purpose:** Define `retrieve_documents(query, top_k)`, which finds the chunks most similar to a question.

**How it works:**
1. **Validate the input.** An empty or whitespace-only query, or `top_k <= 0`, raises a `ValueError`.
2. **Embed the question** with the same model and the same normalization used for the chunks, so that question and chunk scores are directly comparable. Then convert it to `float32`.
3. `search_k = min(top_k, index.ntotal)` never asks for more results than the number of stored vectors.
4. `index.search(query_embedding, search_k)` returns two arrays of shape `(1, search_k)`:
   - `scores`: the similarities
   - `indices`: the vector positions

   Both are already sorted from most to least similar.
5. **Build the results.** The loop skips any `-1` index, which FAISS uses as a placeholder when it has fewer results. It fetches the chunk through `chunk_lookup` and builds a dictionary with `rank` (starting at 1), `chunk_id`, `page`, `score` and `text`.
6. Returns the list of result dictionaries.

**Used by:** the test in Step I-18 and the full RAG pipeline in Step II-8.

**Output:** Nothing is printed (the function is only defined).

In [ ]:
# ============================================================
# 17. DEFINE RETRIEVAL FUNCTION
# ============================================================

def retrieve_documents(
    query,
    top_k=3
):
    """
    Retrieve the top matching chunks using FAISS.

    Returns chunk ID, page, similarity score, and text.
    """

    if not query or not query.strip():
        raise ValueError("The query cannot be empty.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # Convert the question into an embedding.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    # Avoid requesting more results than available vectors.
    search_k = min(top_k, index.ntotal)

    # Search the FAISS vector store.
    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        if idx == -1:
            continue

        chunk = chunk_lookup[int(idx)]

        results.append({
            "rank": rank + 1,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(scores[0][rank]),
            "text": chunk["text"]
        })

    return results

## Step I-18 — Test the Retrieval

**Purpose:** Check retrieval quality on its own, before any language model is involved.

**How it works:**
1. `input()` asks for a test question. Pressing **Enter** with no text skips the test.
2. If a question was typed, `retrieve_documents(test_query, TOP_K)` returns the top chunks.
3. For each result, the cell prints the rank, chunk ID, page, similarity score (4 decimals) and the full chunk text.

**Why it matters:** The language model can only answer from what retrieval finds. If the right chunk is not in this list, the final answer cannot be grounded in it. This is the place to record which chunks come back for each test question and chunk setting.

**Note:** The cell waits for your input. During *Run all*, execution pauses here until you type a question or press Enter.

In [ ]:
# ============================================================
# 18. TEST RETRIEVAL
# ============================================================

print("\n")
print("=" * 70)
print("TESTING VECTOR STORE")
print("=" * 70)

test_query = input(
    "\nEnter a test question "
    "(or press Enter to skip): "
)

if test_query.strip():

    test_results = retrieve_documents(
        test_query,
        TOP_K
    )

    print("\n")
    print("=" * 70)
    print("RETRIEVED CHUNKS")
    print("=" * 70)

    for result in test_results:

        print(f"\nRESULT {result['rank']}")
        print(f"Chunk ID   : {result['chunk_id']}")
        print(f"Page       : {result['page']}")
        print(f"Similarity : {result['score']:.4f}")

        print("-" * 70)

        print(result["text"])

## Step I-19 — Save the FAISS Index

**Purpose:** Save the vector store to a file.

**How it works:**
1. `faiss.write_index(index, "rag_faiss.index")` writes the index, including all stored vectors, as a binary file in the Colab working folder (`/content`).
2. It can be loaded later with `faiss.read_index("rag_faiss.index")` instead of re-computing the embeddings.

**Note:** Colab files are deleted when the runtime ends. Download the file from the Files panel if you need to keep it.

**Output:** The saved file name.

In [ ]:
# ============================================================
# 19. SAVE FAISS INDEX
# ============================================================

FAISS_FILE = "rag_faiss.index"

faiss.write_index(
    index,
    FAISS_FILE
)

print("\nFAISS index saved:", FAISS_FILE)

## Step I-20 — Save the Chunks and Metadata

**Purpose:** Save the chunk texts and their IDs and page numbers.

**How it works:**
1. `open(CHUNKS_FILE, "wb")` opens `rag_chunks.pkl` for **w**riting in **b**inary mode.
2. `pickle.dump(chunks, f)` serializes the whole `chunks` list (the dictionaries with `chunk_id`, `page` and `text`) into the file.
3. Together with the saved index, this file is enough to rebuild `chunk_lookup` and run searches later without re-chunking the PDF. The embedding model is still needed to encode new questions.

**Output:** The saved file name.

In [ ]:
# ============================================================
# 20. SAVE CHUNKS AND METADATA
# ============================================================

CHUNKS_FILE = "rag_chunks.pkl"

with open(CHUNKS_FILE, "wb") as f:

    pickle.dump(
        chunks,
        f
    )

print("Chunks saved:", CHUNKS_FILE)

## Step I-21 — Save the Indexing Configuration

**Purpose:** Record which settings produced this index.

**How it works:**
1. Builds the `config` dictionary with `chunk_size`, `chunk_overlap`, `top_k`, `embedding_model` and `source_document`.
2. `pickle.dump` writes it to `rag_config.pkl`.
3. This supports reproducibility. When several chunk configurations are compared, each saved index can be matched to the exact settings that created it.

**Output:** The saved file name.

In [ ]:
# ============================================================
# 21. SAVE INDEXING CONFIGURATION
# ============================================================

CONFIG_FILE = "rag_config.pkl"

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "source_document": pdf_file
}

with open(CONFIG_FILE, "wb") as f:

    pickle.dump(
        config,
        f
    )

print("Configuration saved:", CONFIG_FILE)

## Step I-22 — Indexing Complete

**Purpose:** Print a final summary of Component I.

**How it works:**
1. Prints the source document, the number of chunks, the vectors stored and the chunk settings. The number of vectors should equal the number of chunks.
2. Lists the four files created: the Excel workbook, `rag_faiss.index`, `rag_chunks.pkl` and `rag_config.pkl`.
3. Confirms that the vector store is ready for retrieval.

**Next:** Component II uses `retrieve_documents`, `embedding_model`, `index`, `chunk_lookup` and `TOP_K` from memory. Run it in the **same session** without restarting the runtime.

In [ ]:
# ============================================================
# 22. INDEXING COMPLETE
# ============================================================

print("\n")
print("=" * 70)
print("INDEXING COMPLETE")
print("=" * 70)

print("Source document :", pdf_file)
print("Number of chunks:", len(chunks))
print("Vectors stored  :", index.ntotal)
print("Chunk size      :", CHUNK_SIZE)
print("Chunk overlap   :", CHUNK_OVERLAP)
print("Embedding model :", EMBEDDING_MODEL_NAME)

print("\nSaved files:")
print("  ", EXCEL_FILE)
print("  ", FAISS_FILE)
print("  ", CHUNKS_FILE)
print("  ", CONFIG_FILE)

print("\nThe vector store is ready for retrieval.")
print(
    "Inspect the downloaded Excel workbook to evaluate "
    "the generated chunks."
)

# Component II: Prompt and Retrieval

## Step II-1 — Import the Required Libraries

**Purpose:** Load the libraries for the language-model part of the pipeline.

**How it works:**
1. `AutoTokenizer` and `AutoModelForSeq2SeqLM` from Hugging Face `transformers` pick the correct tokenizer and model classes automatically from the model name. "Seq2Seq" means an **encoder–decoder** model: it reads an input text (the prompt) and writes a new output text (the answer). FLAN-T5 is this kind of model.
2. `torch` (PyTorch) provides tensors, GPU detection and `no_grad()`.
3. `re` is imported again. It is not used in this component, and the import is harmless.

**Notes:**
- The banner comment says `flan-t5-small`, but Step II-3 actually loads **`google/flan-t5-base`**. The code is what runs.
- This component depends on objects created in Component I. Run Component I first in the same session.

**Output:** Nothing is printed.

In [ ]:
# ============================================================
# RAG PROMPT PROCESSING
# TOKEN-LIMITED CONTEXT, ANSWER GENERATION, AND DIAGNOSTICS
# Model: google/flan-t5-small
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

import torch
import re

## Step II-2 — Check the Device

**Purpose:** Choose where the language model will run.

**How it works:**
1. `torch.cuda.is_available()` returns `True` when the Colab runtime has a GPU.
2. The conditional expression sets `device` to `"cuda"` (GPU) or `"cpu"`.
3. Prints the `RAG PROMPT PROCESSING` banner and the chosen device.

**Tip:** In Colab, a GPU is enabled under *Runtime → Change runtime type*. A GPU makes answer generation faster, but the pipeline also works on the CPU.

**Output:** `Device: cuda` or `Device: cpu`.

In [ ]:
# ============================================================
# 2. CHECK DEVICE
# ============================================================

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("RAG PROMPT PROCESSING")
print("=" * 70)

print("Device:", device)

## Step II-3 — Load the Local LLM

**Purpose:** Load the language model that writes the final answer, along with its token limits.

**How it works:**
1. `LLM_NAME = "google/flan-t5-base"`: an instruction-tuned T5 model with about 250 million parameters.
2. Token settings. A **token** is the unit a language model reads: a word or part of a word.
   - `MAX_INPUT_TOKENS = 512`: the largest prompt this notebook allows. T5 models were pre-trained on 512-token inputs.
   - `MAX_NEW_TOKENS = 32`: the maximum length of the generated answer, so answers stay short.
   - `RESERVED_TOKEN_MARGIN = 16`: a safety buffer kept free inside the 512-token budget (see Step II-6).
3. `AutoTokenizer.from_pretrained` and `AutoModelForSeq2SeqLM.from_pretrained` download the tokenizer and model weights from the Hugging Face Hub the first time, then load them from the cache.
4. `llm.to(device)` moves the model to the GPU or CPU chosen in Step II-2.
5. `llm.eval()` switches the model to inference mode, which turns off dropout so the same input gives the same output.

**Output:** The model name and `Language model loaded.`

In [ ]:
# ============================================================
# 3. LOAD THE LOCAL LLM
# ============================================================

LLM_NAME = "google/flan-t5-base"

MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 32

# Reserve space within the input budget for safety.
RESERVED_TOKEN_MARGIN = 16

print("\nLoading language model:")
print(LLM_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME
)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    LLM_NAME
)

llm = llm.to(device)
llm.eval()

print("Language model loaded.")

## Step II-4 — Define the Token Counting Function

**Purpose:** Define `count_tokens(text)`, which measures exactly how many tokens the model will see for a text.

**How it works:**
1. `tokenizer.encode(text, ...)` converts the text into a list of token IDs.
   - `add_special_tokens=True` includes the end-of-sequence token `</s>` that T5 adds, so the count matches the real model input.
   - `truncation=False` never cuts the text, so the true full length is counted.
2. `len(...)` returns the number of tokens.

**Used by:** the final safety check in Step II-6.

**Output:** Nothing is printed (the function is only defined).

In [ ]:
# ============================================================
# 4. DEFINE TOKEN COUNTING FUNCTION
# ============================================================

def count_tokens(text):

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )

## Step II-5 — Prioritize the Retrieved Documents

**Purpose:** Define `prioritize_documents(question, retrieved_documents)`, which orders the evidence from most to least relevant.

**How it works:**
1. `sorted(..., key=..., reverse=True)` sorts the results by their `"score"`, highest first.
2. `result.get("score", float("-inf"))` places any result without a score at the end.
3. FAISS (Step I-17) already returns results in this order, so the function normally leaves the order unchanged. It is a safeguard that guarantees the best evidence goes into the prompt first.
4. The `question` parameter is accepted but not used. It leaves room for a future re-ranking method.

**Output:** Nothing is printed (the function is only defined).

In [ ]:
# ============================================================
# 5. PRIORITIZE RETRIEVED DOCUMENTS
# ============================================================

def prioritize_documents(question, retrieved_documents):
    """
    Preserve FAISS similarity ranking.

    Assumes higher scores mean greater similarity,
    as with normalized embeddings and IndexFlatIP.
    """

    return sorted(
        retrieved_documents,
        key=lambda result: result.get("score", float("-inf")),
        reverse=True
    )

## Step II-6 — Define the Token-Limited RAG Prompt

**Purpose:** Define `create_rag_prompt(question, retrieved_documents)`. It builds the text sent to the language model, fitting as much retrieved evidence as possible into the 512-token budget.

**How it works:**
1. **Rank the evidence** with `prioritize_documents` (Step II-5).
2. **Instruction.** The earlier instruction is kept as commented-out lines and is not active. The active `instruction` tells the model:
   - use only the supplied evidence
   - match the requested person, position or organization to its explicitly stated role
   - do not guess or use outside knowledge
   - reply exactly `'I cannot find the answer in the provided document.'` when the evidence does not support an answer
3. **Prompt layout:** `Question: …` + instruction + `Evidence:` + *context* + `Answer:`. The question comes first so it can never be cut off.
4. **Measure the fixed parts.** The prefix and suffix are tokenized without special tokens.
5. **Compute the evidence budget:** `available_context_tokens = 512 − prefix − suffix − 16 (margin) − 1 (end token)`. If nothing is left, a `ValueError` asks for a shorter question.
6. **Fill the budget, chunk by chunk** (best first):
   - Label each chunk as `[Page p, Chunk c]` followed by its text, then tokenize it.
   - Count a `"\n\n"` separator before every chunk except the first.
   - If the chunk fits, include it whole. Otherwise keep only its first `tokens_available` tokens, cutting at a token boundary.
   - Decode the kept tokens back to text and subtract the tokens used from the budget.
   - If a chunk had to be cut, stop: the budget is full.
7. **Assemble** the prompt from the prefix, the evidence parts joined by blank lines, and the suffix.
8. **Final check.** `count_tokens(prompt)` re-counts the complete prompt. More than 512 tokens raises a `ValueError`. This check, together with the 16-token margin, is needed because decoding tokens to text and encoding again can change the count slightly.
9. **Diagnostics.** Prints the question, the number of chunks retrieved and included, the final token count, the limit and every piece of selected evidence.
10. Returns the finished prompt string.

**Output:** Nothing is printed (the function is only defined). It prints diagnostics each time it is called.

In [ ]:
# ============================================================
# 6. DEFINE THE TOKEN-LIMITED RAG PROMPT
# ============================================================

def create_rag_prompt(
    question,
    retrieved_documents
):
    """
    Build a prompt that preserves the question and includes
    as much relevant retrieved evidence as the input budget
    allows.

    Context is trimmed only at token boundaries.
    """

    ranked_documents = prioritize_documents(
        question,
        retrieved_documents
    )

    #instruction = (
    #    "Answer the question using only the evidence below. "
    #    "Give a concise answer. Do not invent information. "
    #    'If the evidence does not support an answer, say: '
    #    '"I cannot find the answer in the provided document."'
    #)

    instruction = (
    "Answer the question using only the supplied evidence. "
    "Identify the exact person, position, department, or "
    "organization requested. When the evidence lists multiple "
    "people or departments, match the requested entity to "
    "its explicitly associated name and role. Do not select "
    "a person merely because their name appears near a "
    "matching keyword. Do not guess or use outside knowledge. "
    "For a question asking who holds a position, return the "
    "full name explicitly associated with that position "
    "and organization. If the evidence does not establish "
    "the relationship, respond exactly: "
    "'I cannot find the answer in the provided document.'"
)

    # Keep the question near the beginning.
    prompt_prefix = (
        f"Question: {question}\n\n"
        f"{instruction}\n\n"
        "Evidence:\n"
    )

    prompt_suffix = "\n\nAnswer:"

    # Tokenize the fixed parts first.
    prefix_tokens = tokenizer.encode(
        prompt_prefix,
        add_special_tokens=False
    )

    suffix_tokens = tokenizer.encode(
        prompt_suffix,
        add_special_tokens=False
    )

    # Keep the final prompt safely within the model limit.
    available_context_tokens = (
        MAX_INPUT_TOKENS
        - len(prefix_tokens)
        - len(suffix_tokens)
        - RESERVED_TOKEN_MARGIN
        - 1  # Account for a possible special token.
    )

    if available_context_tokens <= 0:
        raise ValueError(
            "The question and instructions leave no room "
            "for retrieved evidence. Shorten the question."
        )

    selected_parts = []
    remaining_tokens = available_context_tokens

    for result in ranked_documents:

        if remaining_tokens <= 0:
            break

        source_text = (
            f"[Page {result['page']}, "
            f"Chunk {result['chunk_id']}]\n"
            f"{result['text']}"
        )

        # Tokenize each source separately.
        source_tokens = tokenizer.encode(
            source_text,
            add_special_tokens=False
        )

        separator_tokens = (
            tokenizer.encode(
                "\n\n",
                add_special_tokens=False
            )
            if selected_parts
            else []
        )

        tokens_available = (
            remaining_tokens - len(separator_tokens)
        )

        if tokens_available <= 0:
            break

        # Include the full chunk when it fits.
        # Otherwise, retain the beginning of the chunk.
        included_tokens = source_tokens[
            :tokens_available
        ]

        if not included_tokens:
            continue

        included_text = tokenizer.decode(
            included_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        selected_parts.append(included_text)

        remaining_tokens -= (
            len(separator_tokens)
            + len(included_tokens)
        )

        # Stop if this chunk had to be truncated.
        if len(included_tokens) < len(source_tokens):
            break

    context = "\n\n".join(selected_parts)

    prompt = (
        prompt_prefix
        + context
        + prompt_suffix
    )

    # Final check using the actual tokenizer.
    final_token_count = count_tokens(prompt)

    if final_token_count > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Prompt contains {final_token_count} tokens, "
            f"exceeding the {MAX_INPUT_TOKENS}-token limit. "
            "Reduce the context budget."
        )

    print("\n--- PROMPT DIAGNOSTICS ---")
    print("Original question:", question)
    print("Retrieved chunks:", len(retrieved_documents))
    print("Chunks included:", len(selected_parts))
    print("Final prompt tokens:", final_token_count)
    print("Input token limit:", MAX_INPUT_TOKENS)

    print("\n--- SELECTED EVIDENCE ---")

    for i, part in enumerate(selected_parts, start=1):
      print(f"\nEvidence {i}:")
      print(part)

    return prompt

## Step II-7 — Define the Answer Generation Function

**Purpose:** Define `generate_answer(prompt, max_new_tokens)`, which runs FLAN-T5 on the prompt and returns the answer text.

**How it works:**
1. `tokenizer(prompt, return_tensors="pt", truncation=False)` converts the prompt into PyTorch tensors (`input_ids`, `attention_mask`) without silently cutting it.
2. `input_length` is read from the tensor shape and printed. If it exceeds `MAX_INPUT_TOKENS`, a `ValueError` is raised instead of letting the model drop text.
3. The dictionary comprehension moves every tensor to the same `device` as the model.
4. `torch.no_grad()` turns off gradient tracking, which is only needed for training. This saves memory and time.
5. `llm.generate(...)` writes the answer:
   - `do_sample=False` with `num_beams=4` uses **beam search**. The model keeps the 4 most likely partial answers at every step and returns the best complete one. With no random sampling, the same prompt always gives the same answer.
   - `early_stopping=True` ends the search once all beams are finished.
   - `max_new_tokens` caps the answer length (32 tokens).
6. `tokenizer.decode(outputs[0], skip_special_tokens=True)` turns the output tokens back into plain text. `.strip()` removes surrounding spaces.
7. An empty answer is replaced with a message asking you to inspect the prompt and evidence.

**Output:** Nothing is printed (the function is only defined). It prints generation diagnostics each time it is called.

In [ ]:
# ============================================================
# 7. DEFINE ANSWER GENERATION FUNCTION
# ============================================================

def generate_answer(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS
):
    """
    Generate an answer without silently truncating the prompt.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False
    )

    input_length = inputs["input_ids"].shape[1]

    print("\n--- GENERATION DIAGNOSTICS ---")
    print("Actual input tokens:", input_length)
    print("Maximum input tokens:", MAX_INPUT_TOKENS)

    if input_length > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Input contains {input_length} tokens, "
            f"but the model limit is {MAX_INPUT_TOKENS}."
        )

    # Move model inputs to the selected device.
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate deterministically.
    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    if not answer:
        return (
            "The model generated an empty answer. "
            "Inspect the prompt and retrieved evidence."
        )

    return answer

## Step II-8 — Define the Complete RAG Query Function

**Purpose:** Define `rag_query(question, top_k)`, which connects retrieval, prompt building and generation into one **RAG** (Retrieval-Augmented Generation) call.

**How it works (matches the `STEP` comments in the code):**
1. **Validate:** strip the question. An empty question returns a "Please enter a valid question." message.
2. **Retrieve:** `retrieve_documents(question, top_k)` from Step I-17 gets the most similar chunks.
3. **No evidence:** if nothing is retrieved, return the fallback sentence "I cannot find the answer in the provided document."
4. **Build the prompt:** `create_rag_prompt` (Step II-6).
5. **Generate:** `generate_answer` (Step II-7).
6. **Empty answer:** replace it with a message asking you to review the evidence.
7. **Return** a 3-item tuple `(answer, retrieved_documents, prompt)`, so the caller can show the answer together with its sources and the exact prompt.

**Output:** Nothing is printed (the function is only defined).

In [ ]:
# ============================================================
# 8. DEFINE COMPLETE RAG QUERY FUNCTION
# ============================================================


def rag_query(question, top_k=3):
    """
    General-purpose RAG question-answering function.

    Workflow:
    1. Retrieve relevant document chunks.
    2. Build a token-limited prompt.
    3. Generate an answer using the language model.
    4. Return the answer, retrieved sources, and prompt.
    """

    # STEP 1: Validate the question.
    question = question.strip()

    if not question:
        return (
            "Please enter a valid question.",
            [],
            ""
        )

    # STEP 2: Retrieve relevant document chunks.
    retrieved_documents = retrieve_documents(
        question,
        top_k=top_k
    )

    # STEP 3: Handle cases where no evidence is retrieved.
    if not retrieved_documents:
        return (
            "I cannot find the answer in the provided document.",
            [],
            ""
        )

    # STEP 4: Construct the token-limited RAG prompt.
    prompt = create_rag_prompt(
        question,
        retrieved_documents
    )

    # STEP 5: Generate an answer from the retrieved evidence.
    answer = generate_answer(prompt)

    # STEP 6: Handle an empty model response.
    if not answer or not answer.strip():
        answer = (
            "The model could not generate an answer. "
            "Please review the retrieved evidence."
        )

    # STEP 7: Return the results.
    return (
        answer.strip(),
        retrieved_documents,
        prompt
    )

## Step II-9 — Continuous Question-Answer Loop

**Purpose:** Provide an interactive session for asking questions about the indexed document.

**How it works:**
1. Prints the usage instructions and creates `last_answer`, `last_sources` and `last_prompt` to remember the most recent query. `last_answer` is stored but not displayed by any command.
2. `while True:` repeats until you leave. Each round, `input("Question: ").strip()` reads your text. The checks run in this order, and the command words ignore letter case:
   - `exit` → print "RAG session terminated." and `break` out of the loop.
   - empty input → ask again (`continue`).
   - `sources` → print the rank, page, chunk, similarity and text of every chunk from the last query.
   - `prompt` → print the full prompt sent to the model in the last query.
   - any other text is treated as a question.
3. **Process a question** inside `try:`:
   - `rag_query(user_question, top_k=TOP_K)` runs the pipeline. The diagnostics from Steps II-6 and II-7 print first.
   - The results are saved in the `last_*` variables.
   - The cell prints three blocks: `RAG ANSWER`, `RETRIEVED SOURCES` and `RAG PROMPT SENT TO LLM`.
4. `except Exception as e:` prints the error message and keeps the loop running, so one failed question does not end the session.

**Note:** This cell keeps running, waiting for input, until you type `exit`. Other cells cannot run while it is waiting.

In [ ]:
# ============================================================
# 9. CONTINUOUS QUESTION-ANSWER LOOP
# ============================================================

print("\n")
print("=" * 70)
print("CONTINUOUS RAG QUESTION ANSWERING")
print("=" * 70)

print("Ask questions about the indexed document.")
print("Type 'exit' to terminate.")
print("Type 'sources' to display sources from the last query.")
print("Type 'prompt' to display the last generated prompt.")


# Store the most recent query results.
last_answer = None
last_sources = []
last_prompt = None


while True:

    print("\n")

    user_question = input(
        "Question: "
    ).strip()

    # --------------------------------------------------------
    # EXIT CONDITION
    # --------------------------------------------------------

    if user_question.lower() == "exit":

        print("\nRAG session terminated.")
        break

    # --------------------------------------------------------
    # IGNORE EMPTY QUESTIONS
    # --------------------------------------------------------

    if not user_question:

        print("Please enter a question.")
        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: SOURCES
    # --------------------------------------------------------

    if user_question.lower() == "sources":

        if not last_sources:
            print("No previous query sources are available.")
            continue

        print("\n" + "=" * 70)
        print("SOURCES FROM LAST QUERY")
        print("=" * 70)

        for result in last_sources:

            print(
                f"\nRank {result['rank']} | "
                f"Page {result['page']} | "
                f"Chunk {result['chunk_id']} | "
                f"Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: PROMPT
    # --------------------------------------------------------

    if user_question.lower() == "prompt":

        if last_prompt is None:
            print("No previous query prompt is available.")
            continue

        print("\n" + "=" * 70)
        print("LAST RAG PROMPT")
        print("=" * 70)

        print(last_prompt)

        continue

    # --------------------------------------------------------
    # PROCESS QUESTION
    # --------------------------------------------------------

    try:

        answer, retrieved_documents, prompt = rag_query(
            user_question,
            top_k=TOP_K
        )

        last_answer = answer
        last_sources = retrieved_documents
        last_prompt = prompt

        # ----------------------------------------------------
        # DISPLAY ANSWER
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG ANSWER")
        print("=" * 70)

        print(answer)

        # ----------------------------------------------------
        # DISPLAY RETRIEVED SOURCES
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RETRIEVED SOURCES")
        print("=" * 70)

        for result in retrieved_documents:

            print(
                f"\n[{result['rank']}] "
                f"Page {result['page']} "
                f"| Chunk {result['chunk_id']} "
                f"| Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        # ----------------------------------------------------
        # DISPLAY PROMPT
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG PROMPT SENT TO LLM")
        print("=" * 70)

        print(prompt)

    except Exception as e:

        print("\nERROR:")
        print(str(e))

        print("\nThe question could not be processed.")

## Step II-10 — End

**Purpose:** Mark the end of the program.

**How it works:** Runs after you type `exit` in Step II-9 and prints the `RAG PROGRAM FINISHED` banner. The FAISS index, chunks and models stay in memory, so Step II-9 can be run again for a new session without repeating Component I.

**Output:** The closing banner.

In [ ]:
# ============================================================
# 10. END
# ============================================================

print("\n")
print("=" * 70)
print("RAG PROGRAM FINISHED")
print("=" * 70)